# CGDD-Net — Kaggle GPU Controlled Ablations · First Batch

**2026-10-20 | NEW evaluation protocol / 新实验协议**

This notebook reconstructs **fresh, legally sourced** CHASE_DB1 or STARE data and performs a small software pilot, then (by default) starts **one controlled 100-epoch training run** on Kaggle GPU. It **does not** recover historical experiments or replace paper Table 7/10. A separate saved run per variant is recommended.

**数据许可和结果声明：**只从官方机构档案下载，或使用你已合法取得并上传至私人 Kaggle Input 的官方 ZIP/TAR；不使用未经核实的镜像，不重分发原始图像。首次运行的是全新的固定划分，不应称为原稿结果复现。所有变体必须共用同一个划分、训练预算和随机种子。

官方来源：[CHASE_DB1 — Kingston University](https://researchinnovation.kingston.ac.uk/en/datasets/chasedb1-retinal-vessel-reference-dataset-4/)（CC BY 4.0） · [STARE — Clemson University](https://cecas.clemson.edu/~ahoover/stare/probing/)（使用原站提供的20张图像与 Adam Hoover 标注，须遵循其使用条款并引用原论文） · [Kaggle Notebook docs](https://www.kaggle.com/docs/notebooks).

**Before running / 启动前**：Kaggle Notebook → Settings → Accelerator: GPU, Internet: On. Keep the notebook **private** if attaching non-redistributable datasets. Outputs containing reconstructed probabilities should also be treated as research records.

## 1 · Experiment settings / 实验设置

Start with `CHASEDB1` and `RUN_MODE="first_run"`. The first GPU trial is marked as a smoke test; then the actual new-protocol run uses 100 epochs and **1024 sampled 64×64 training patches per epoch** (far fewer than the manuscript's 150,000). This is a *new equal-budget controlled comparison*, **not** a reproduction of the original model's published scores.

`RUN_MODE="pilot_only"`: prepare data and one short smoke run. `"first_run"`: pilot + full CGDD-Net. `"first_batch"`: pilot + five core variants sequentially, which may exceed a free GPU session; prefer multiple sessions.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, time

DATASET = "CHASEDB1"            # or "STARE"
SPLIT_SEED = 20261020          # ONE split for ALL variants and seeds
SEED = 42
RUN_MODE = "first_run"         # pilot_only | first_run | first_batch
CORE_VARIANTS = ["full", "independent_detail", "fixed_offsets", "uniform_samg", "unet_same_width"]

# Controlled budget (NEW, same for all variants). Full paper = 150000 patches/epoch.
EPOCHS = 100
PATCHES_PER_EPOCH = 1024       # change only BEFORE comparing any variants
TRAIN_BATCH_SIZE = 8           # lower to 4 if OOM; same across all variants

BRANCH = "experiments/pr-controlled-ablations-20261020"
# Pin the experimental source commit before running your complete model matrix.
PINNED_COMMIT = "b44d0c8f5972aae470beefe69b783273d70f061f"  # CI-tested source SHA

# Kaggle working outputs are versionable; temp contains data extracted from provider archives.
REPO_DIR = Path("/kaggle/working/CGDD-Net")
DATA_ROOT = Path("/kaggle/temp/cgdd_data")
RUNS_ROOT = Path("/kaggle/working/cgdd_pr_runs")
CONFIG_DIR = Path("/kaggle/working/pr_configs")
ARCHIVE_DIR = Path("/kaggle/input")
ATTACHED_SOURCE_DIR = None     # e.g. "/kaggle/input/my-private-university-archive" if Kaggle unzips it

assert DATASET in {"CHASEDB1", "STARE"}
assert RUN_MODE in {"pilot_only", "first_run", "first_batch"}
print("Dataset:", DATASET, "mode:", RUN_MODE, "seed:", SEED)
print("Budget:", EPOCHS, "epochs x", PATCHES_PER_EPOCH, "patches/epoch; batch", TRAIN_BATCH_SIZE)
print("IMPORTANT: This is a NEW protocol, not the author-reported original scores.")

## 2 · Verify Kaggle GPU & clone source / 检查 GPU 和源代码

This environment has no direct Kaggle access from ChatGPT. The first real training starts only when **you run these notebook cells in your Kaggle account**; no training is claimed from creating this file.

In [ ]:
import torch
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("CUDA availability:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle: Settings > Accelerator > GPU, then restart the session")
print("GPU:", torch.cuda.get_device_name(0),
      "memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:1400])

In [ ]:
url = "https://github.com/lixincheng-xcl/CGDD-Net.git"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)], check=True)
if PINNED_COMMIT:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", PINNED_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Pinned experimental code:", commit)
assert (REPO_DIR / "experiments" / "kaggle_data.py").is_file(), "Use the new experiment branch/commit"
# Kaggle already includes torch/numpy/scipy/scikit-learn in most GPU images.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps", "-q"], check=True)
for package in ("PIL", "numpy", "sklearn", "scipy"):
    __import__(package)
print("Repository and dependencies: OK")

## 3 · Attach original provider archives or download them / 数据合法获取

**Option A (default)**: enable Internet and let the script try the university-hosted original files. Kingston sometimes denies scripted downloads; this is not a reason to use random mirrors.

**Option B**: download original data yourself from the official provider after accepting its conditions. Create a **private Kaggle Dataset** containing `CHASEDB1.zip`, or `stare-images.tar` + `labels-ah.tar`; attach it to the Notebook under **Add Input**. Kaggle can **automatically unpack archives**; the notebook accepts either intact ZIP/TAR filenames or the original extracted provider files, without re-uploading the originals publicly. Set `ATTACHED_SOURCE_DIR` if auto-detection sees multiple candidates.

The processing code checks the provider archive format, **28 complete left/right CHASE pairs**, or **20 STARE IDs with matching observer labels**. It preserves the official observer labels but converts image files losslessly to PNG for consistent loading. The first-batch archives do *not* provide authoritative pixel-level FOV masks in the chosen packaging; the protocol explicitly evaluates the full rectangle instead of fabricating FOV masks. This differs from masked protocols and must be reported.

In [ ]:
def exactly_one_archived_file(name):
    matches = sorted(ARCHIVE_DIR.rglob(name)) if ARCHIVE_DIR.is_dir() else []
    if len(matches) > 1:
        raise RuntimeError(f"Several {name} archives attached; keep only the intended provider file: {matches}")
    return matches[0] if matches else None

def auto_detect_extracted():
    if ATTACHED_SOURCE_DIR:
        folder = Path(ATTACHED_SOURCE_DIR)
        if not folder.is_dir():
            raise FileNotFoundError(folder)
        return folder
    if not ARCHIVE_DIR.is_dir():
        return None
    candidates = []
    for root in sorted(x for x in ARCHIVE_DIR.iterdir() if x.is_dir()):
        filenames = [p.name.lower() for p in root.rglob("*") if p.is_file()]
        if DATASET == "CHASEDB1":
            found = sum(name.startswith("image_") and name.endswith((".jpg", ".jpeg"))
                        and "_1stho" not in name for name in filenames)
            matched = sum("_1stho." in name for name in filenames)
            suitable = (found == 28 and matched == 28)
        else:
            found = sum(name.startswith("im") and name.endswith((".ppm", ".ppm.gz"))
                        and ".ah." not in name for name in filenames)
            matched = sum(".ah.ppm" in name for name in filenames)
            suitable = (found == 20 and matched == 20)
        if suitable:
            candidates.append(root)
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple extracted provider archives detected: {candidates}; set ATTACHED_SOURCE_DIR")
    return candidates[0] if candidates else None

args = [sys.executable, "-m", "experiments.kaggle_data", "--dataset", DATASET,
        "--output", str(DATA_ROOT), "--split-seed", str(SPLIT_SEED)]
if DATASET == "CHASEDB1":
    archive = exactly_one_archived_file("CHASEDB1.zip")
    if archive:
        args += ["--chase-zip", str(archive)]
    elif extracted := auto_detect_extracted():
        args += ["--source-folder", str(extracted)]
else:
    images_tar = exactly_one_archived_file("stare-images.tar")
    labels_tar = exactly_one_archived_file("labels-ah.tar")
    if bool(images_tar) != bool(labels_tar):
        raise RuntimeError("STARE needs BOTH original provider tar files")
    if images_tar:
        args += ["--stare-images", str(images_tar), "--stare-labels", str(labels_tar)]
    elif extracted := auto_detect_extracted():
        args += ["--source-folder", str(extracted)]

subprocess.run(args, cwd=REPO_DIR, check=True)
train_json = DATA_ROOT / "manifests" / DATASET / "train.json"
test_json = DATA_ROOT / "manifests" / DATASET / "test.json"
provenance = json.loads((test_json.parent / "provenance.json").read_text())
print(json.dumps(provenance, indent=2, ensure_ascii=False))
# Preserve metadata only, not the image bytes, under Kaggle versioned outputs.
meta_dir = Path("/kaggle/working/pr_split_records") / DATASET
meta_dir.mkdir(parents=True, exist_ok=True)
for f in (train_json, test_json, test_json.parent / "provenance.json"):
    shutil.copy2(f, meta_dir / f.name)
assert provenance["training_count"] == (20 if DATASET == "CHASEDB1" else 10)
assert provenance["test_count"] == (8 if DATASET == "CHASEDB1" else 10)

## 4 · Inspect original cases and split / 核对图像、标签和划分

View image/label alignment **before** training; this is a data integrity check, not performance-driven test selection. CHASE test subjects must be distinct from train subjects. Save generated split metadata and its archive hashes to reproduce all comparisons. Never change the split after inspecting model performance.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from cgddnet.data import load_manifest, load_case, validate_splits
train_records = load_manifest(train_json, DATA_ROOT)
test_records = load_manifest(test_json, DATA_ROOT)
print("Split audit:", validate_splits({"train": train_records, "test": test_records}))
print("Training IDs:", [r["id"] for r in train_records])
print("Test IDs:", [r["id"] for r in test_records])
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, example in enumerate((train_records[0], test_records[0])):
    with Image.open(example["image"]) as image:
        axes[row, 0].imshow(image)
    with Image.open(example["label"]) as image:
        axes[row, 1].imshow(image, cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title(f'{"Train" if row == 0 else "Test"}: {example["id"]}')
    axes[row, 1].set_title("Original first-observer vessel label")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("FOV policy: FULL RECTANGLE; do NOT compare against FOV-masked paper results without harmonization.")

## 5 · Create new equal-budget configuration / 固定训练预算

The reference repo documents 100 epochs, 150,000 patch samples/epoch, and training batch 64 for the manuscript. Kaggle free GPU is unlikely to complete a full 5×2×3 rerun at that budget. We therefore explicitly predeclare **a different, equal-budget comparison protocol** and keep every new result separate. The test set is evaluated only after the final epoch; there is no test-based model selection. To claim exact historical reproduction, the original manifests and historical budget would be required.

All five models use identical augmentation, patch sizes, loss, learning rate, optimizer, inference reconstruction, seed, epoch count and sampled-patch budget.

In [ ]:
cfg = json.loads((REPO_DIR / "configs/default.json").read_text())
cfg["training"].update({"epochs": EPOCHS, "patches_per_epoch": PATCHES_PER_EPOCH,
                        "batch_size": TRAIN_BATCH_SIZE, "early_stopping": 0,
                        "num_workers": 0})
cfg["inference"].update({"patch_size": 64, "stride": 16, "batch_size": 8, "threshold": 0.5})
cfg["description"] = (
    "NEW Kaggle 2026 controlled comparison: not historical results; "
    f"fixed {EPOCHS} epochs, {PATCHES_PER_EPOCH} patches/epoch, "
    f"training batch={TRAIN_BATCH_SIZE}; full rectangle evaluation (no official FOV masks)."
)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
config_file = CONFIG_DIR / f"{DATASET.lower()}_kaggle_budget.json"
config_file.write_text(json.dumps(cfg, indent=2) + "\n")
print("Configuration:", config_file)
print("Training iterations per model:", EPOCHS * ((PATCHES_PER_EPOCH + TRAIN_BATCH_SIZE - 1)//TRAIN_BATCH_SIZE))
print("IMPORTANT: use this SAME config for every comparison; DO NOT change mid-series.")

## 6 · Real GPU smoke test / GPU 烟雾测试

Run one epoch with 32 sampled patches; the script flags it `smoke_test_only=True`. GPU memory and the first forward/backward are exercised. Any AUC printed by the smoke test is not valid benchmark evidence. Never add it to the paper.

In [ ]:
pilot_cfg = json.loads(json.dumps(cfg))
pilot_cfg["training"]["batch_size"] = min(TRAIN_BATCH_SIZE, 4)
pilot_config = CONFIG_DIR / f"{DATASET.lower()}_pilot.json"
pilot_config.write_text(json.dumps(pilot_cfg, indent=2) + "\n")
pilot_run = RUNS_ROOT / "software_smoke" / DATASET / f"seed_{SEED}"
# Only one test image in the software pilot, to avoid spending the GPU quota on
# a full 8/10-image sliding-window test that is not scientifically reported.
pilot_doc = json.loads(test_json.read_text())
pilot_doc["records"] = pilot_doc["records"][:1]
pilot_doc["split"] = "software_smoke_test_one_image"
pilot_test_json = CONFIG_DIR / f"{DATASET.lower()}_pilot_test_one_image.json"
pilot_test_json.write_text(json.dumps(pilot_doc, indent=2) + "\n")
command = [sys.executable, "-m", "experiments.train_controlled", "--config", str(pilot_config),
           "--variant", "full", "--train-manifest", str(train_json), "--test-manifest", str(pilot_test_json),
           "--data-root", str(DATA_ROOT), "--seed", str(SEED),
           "--output", str(pilot_run), "--device", "cuda",
           "--epochs-override", "1", "--patches-override", "32"]
if (pilot_run / "experiment_summary.json").is_file():
    print("Previously completed software smoke test found:", pilot_run)
else:
    t0 = time.time()
    subprocess.run(command, cwd=REPO_DIR, check=True)
    print("Software smoke test seconds:", round(time.time()-t0, 1))
print("SMOKE TEST ONLY; its measured scores must never be reported as retinal benchmark results.")

## 7 · Start first core training / 启动第一批正式对照

With `RUN_MODE="first_run"` the notebook trains **CGDD-Net full** at the fixed 100-epoch/1024-patches-per-epoch Kaggle budget. Save its Kaggle Notebook version/output afterward. Then switch to `"first_batch"` to train the remaining four variants **one at a time**; the first completed one is skipped when restored. To resume a partial run, restore the whole previous run folder (including `latest.pt`, config, manifests, log) from the previous notebook's **Output added as Input**.

**No code can launch a Kaggle GPU from this chat session without your Kaggle account access.** Running this cell in Kaggle is the start of the actual experiment.

In [ ]:
# Optional: enter an attached previous Notebook Output folder containing the SAME
# cgdd_pr_runs directory, to resume across Kaggle sessions. Leave None otherwise.
RESTORE_FROM = None       # e.g. "/kaggle/input/my-previous-notebook-output/cgdd_pr_runs"
if RESTORE_FROM:
    old = Path(RESTORE_FROM)
    if not old.is_dir():
        raise FileNotFoundError(old)
    for source in old.glob("*/*/seed_*"):
        target = RUNS_ROOT / source.relative_to(old)
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, target)
    print("Restored previous checkpoint folders from", old)

if RUN_MODE == "pilot_only":
    variants = []
elif RUN_MODE == "first_run":
    variants = ["full"]
else:
    variants = CORE_VARIANTS

for variant in variants:
    run_dir = RUNS_ROOT / DATASET / variant / f"seed_{SEED}"
    if (run_dir / "experiment_summary.json").is_file():
        print(f"SKIP completed: {run_dir}")
        continue
    cmd = [sys.executable, "-m", "experiments.train_controlled",
           "--config", str(config_file), "--variant", variant,
           "--train-manifest", str(train_json), "--test-manifest", str(test_json),
           "--data-root", str(DATA_ROOT), "--output", str(run_dir),
           "--seed", str(SEED), "--device", "cuda"]
    if (run_dir / "latest.pt").is_file():
        cmd.append("--resume")
    print("\nStarting controlled new-protocol training:", DATASET, variant, SEED, flush=True)
    print("Command:", " ".join(cmd))
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    print("Completed", variant, "hours:", round((time.time()-t0)/3600, 2))

print("Completed model variants available:", [p.parent.parent.name for p in RUNS_ROOT.glob(f"{DATASET}/*/seed_{SEED}/experiment_summary.json")])

## 8 · Inspect outputs & save your Kaggle version / 检查并保存结果

For each completed real run, `experiment_summary.json`, `history.csv`, `final.pt`, `test/per_image.csv`, predictions and logs are under `/kaggle/working/cgdd_pr_runs/`. These are **new** dataset-specific run results, not the historical Table 7.

The offline files extracted from official providers remain under `/kaggle/temp`, rather than being redistributed in `/kaggle/working` Notebook output. The visible `pr_split_records` stores JSON manifests only, without original images. Do not publish private Kaggle source data or outputs without verifying source permissions.

After one full run, click **Save Version** on Kaggle and make the previous Notebook Output an Input for the next session before switching variant. Kaggle session limits can still interrupt training mid-epoch; `latest.pt` is saved at the end of each completed epoch.

In [ ]:
for f in sorted(RUNS_ROOT.glob(f"{DATASET}/*/seed_{SEED}/experiment_summary.json")):
    summary = json.loads(f.read_text())
    m = summary["test_scores"]["metrics"]
    print(f.parent.parent.name, "seed", SEED,
          "AUC", round(m["AUC"]["mean"],6) if m["AUC"]["mean"] is not None else None,
          "F1", round(m["F1"]["mean"],6) if m["F1"]["mean"] is not None else None,
          "params", summary["parameter_count"],
          "source", f)
print("Run directory:", RUNS_ROOT)
print("Split provenance (can be archived):", meta_dir)
print("Only include results from fully completed, non-smoke runs and a fixed experimental protocol.")

## 9 · Extension plan / 后续扩展

**Phase 1**: CHASE_DB1, seed 42, full → independent_detail → fixed_offsets → uniform_samg → unet_same_width. **Phase 2**: the same five models on STARE at the *same* new 10/10 split. **Phase 3**: repeat with independently predeclared seeds 2026 and 3407, without changing model/budget/split. Later add PA-Net/FSG-Net only from verified implementations and evaluate full vs budget-matched U-Net.

Once all three seeds are complete, run `experiments/collect_results.py --require-seeds 42 2026 3407`; report between-seed SD and per-image observations separately, and use the statistical tools only on correct paired units. Three training seeds give limited inferential power. Actual manuscript Table 7 or Table 10 cannot be retrospectively fixed by this new experiment.

**Important**: see the accompanying `docs/kaggle_first_batch.md` for checkpoints, data permissions, MANIFEST policy, and Kaggle GPU quota risks.